# Pattern 06: Multi-query decomposition

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for both LLM calls (the
decomposition and the final answer) and the embedding step. Like HyDE, real cost is roughly double
a single-query pattern's, plus retrieval cost scales with however many sub-queries the LLM
produces (up to 3, per `prompts/multi_query_prompt.txt`). Section 7 is a PENDING placeholder
awaiting a real-embeddings-and-LLM run.


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

Multi-query decomposition asks the LLM to rewrite the question into up to 3 alternative search
queries covering different phrasings or sub-aspects (`prompts/multi_query_prompt.txt`, JSON
output). Each sub-query is embedded and searched independently; results are merged by **best rank
across sub-queries** -- a chunk that any sub-query ranked highly ends up ranked highly in the
merged list, regardless of how the others ranked it. The real answer is generated afterward from
the merged retrieval, using the held-constant generation prompt (R4) as usual.


## 2. When to use it

- Questions are ambiguous or compound, covering more than one distinct sub-topic that a single
  embedding vector might not represent well simultaneously
- You want retrieval robustness against a single bad query phrasing -- if one sub-query happens to
  embed poorly, the other 1-2 can still surface the right chunks
- You can afford 2 LLM calls plus multiple retrieval calls per query


## 3. When NOT to use it

- Questions are already simple and unambiguous -- decomposition adds cost and latency for no
  retrieval benefit (the LLM will often just return the original question as the single
  sub-query, per `recipes/multi_query.py`'s defensive fallback)
- Cost/latency budget only allows 1 LLM call and 1 retrieval pass per query
- The decomposition LLM call itself can fail to produce valid JSON -- `_parse_subqueries()`
  degrades gracefully to the original question on any parse failure, but that means a badly
  configured or unreliable model silently loses the benefit of decomposition rather than erroring
  loudly


## 4. Implementation

In [3]:
from recipes.multi_query import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)
print("input_tokens (summed across both LLM calls):", sample.input_tokens)


retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#2', 'arxiv:2601.02404#1']
answer: PCEVAL stands for Physical Computing Evaluation. It is a benchmark designed for fully automatic evaluation of the capabilities of Large Language Models (LLMs) in both the logical and physical aspects of physical computing projects, particularly involving hardware like Arduino systems [arxiv:2601.02404#0].
input_tokens (summed across both LLM calls): 1318


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="06_multi_query",
    judges_enabled=True,
)


=== 06_multi_query (n=18) ===
  hit@3: 1.000  [95% CI 1.000, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.917  [95% CI 0.806, 1.000]
  faithfulness: 0.500  [95% CI 0.278, 0.722]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 0.653  [95% CI 0.509, 0.787]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 5844.4
  p95_latency_ms: 13618.1
  usd_per_query: $0.00993
  eval_usd: $0.1787


## 6. Example query walkthrough

One example per eval-set category, showing the merged-by-best-rank retrieved chunks and the
(mocked) final answer. Under mock, `MockLLM`'s canned decomposition response is not valid
sub-query JSON, so `_parse_subqueries()` falls back to the original question as the only
sub-query -- meaning this mock run exercises the fallback path, not true multi-query fan-out; a
real run with a real LLM is needed to see genuine 2-3-way decomposition.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.02404#2', 'arxiv:2601.02404#1']
A: PCEVAL stands for Physical Computing Evaluation, which is a benchmark designed for evaluating the physical computing capabilities of large language models (LLMs) in both logical and physical aspects of projects involving hardware and software interaction [arxiv:2601.02404#0].



--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: Repeated image generations from the same text prompt in diffusion models end up looking very similar due to a phenomenon called mode collapse. This occurs because the models tend to produce nearly identical outputs across different random seeds when given the same prompt, limiting the diversity of generated images. The initial noise inputs play a crucial role, but simply relying on random seeds often results in low variation. Thus, the model collapses to a few dominant modes of output rather than exploring the full range of possible images for the prompt [arxiv:2601.00090#0, #1].



--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00129#0', 'arxiv:2601.00130#2', 'arxiv:2601.00129#2']
A: The two photonics-AI-systems papers focus on complementary aspects of the challenge of realizing large-scale photonics-empowered AI systems:

1. The first paper ([arxiv:2601.00129]) emphasizes the need for **physical design automation and system-algorithm co-exploration**. It highlights the importance of scalable, manufacturable photonic and electronic-photonic physical design automation (EPDA) to handle complex photonic integrated circuits (PICs) and electronic-photonic integrated circuits (EPICs). It also stresses the necessity of co-optimizing system architecture and learning algorithms while accounting for physical non-idealities, control/calibration limits, and packaging/interface costs. This paper presents a cross-layer toolchain including SimPhony,

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00907#0', 'arxiv:2601.00907#2', 'arxiv:2601.00907#1']
A: The paper titled "Placenta Accreta Spectrum Detection using Multimodal Deep Learning" addresses diagnosing a pregnancy complication using deep learning. It focuses on detecting Placenta Accreta Spectrum (PAS), a life-threatening obstetric condition, by developing a multimodal deep learning framework that integrates 3D MRI and 2D Ultrasound scans to improve diagnostic accuracy and patient outcomes [arxiv:2601.00907#0, #2].



## 7. Where this pattern FAILS

**PENDING: real findings from a one-off real-embeddings run.** A real `OPENAI_API_KEY` was not yet
available in the environment when this notebook was authored. This section will be replaced with a
static table of genuine hit@k/mrr failures (matching the format used in `02_bm25.ipynb`/
`04_rerank.ipynb` section 7), computed via a real, uncommitted exploratory run once a key is
available. The claim will be labeled with the date it was run
and its actual dollar cost, and will not be presented as live-executed cell output, to avoid
implying a mock re-run reproduces it (see this notebook's top-of-file disclaimer).


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal multi-query retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.multi_query import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
